###Considerações:

1 - Não foi usada a arquitetura do exercicio anterior, pois são propostas diferentes.

2 - Em Python quando usa o head() ele ja pega as 5 primeiras linhas ou a quantidade que informar, mas estou usando ROW_NUMBER() para fazer ranking pois o caso de uso foi solicitado para fazer com SQL.

In [ ]:
import json
import re
import unicodedata
from datetime import datetime, timezone
from typing import Dict, Iterable, List

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

# ========== CONFIGURAÇÕES ==========
PROJECT_ID = "elet-dados-pessoas-dev"
DATASET_ID = "ds_ti_swapi_raw"
TABLE_PREFIX = "tb_ti_swapi_raw"
WRITE_DISPOSITION = "WRITE_TRUNCATE"
ENFORCE_ALL_STRING = True

# Endpoints
ENDPOINTS: Dict[str, str] = {
    "films": "https://swapi.dev/api/films/",
    "people": "https://swapi.dev/api/people/",
    "planets": "https://swapi.dev/api/planets/",
    "species": "https://swapi.dev/api/species/",
    "starships": "https://swapi.dev/api/starships/",
    "vehicles": "https://swapi.dev/api/vehicles/",
}


# Cliente BigQuery
bq_client = bigquery.Client(project=PROJECT_ID)


# ========== UTILITÁRIOS HTTP (RETRY & PAGINAÇÃO) ==========
def build_session_with_retry() -> requests.Session:
    """Cria uma sessão requests com retentativas exponenciais para lidar com 429/5xx."""
    retry = Retry(
        total=6,
        read=6,
        connect=6,
        status=6,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=20)
    session = requests.Session()
    session.headers.update({"User-Agent": "colab-enterprise-swapi-loader/1.0"})
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def fetch_paginated(session: requests.Session, url: str) -> Iterable[dict]:
    """Itera por todas as páginas do SWAPI, rendendo cada item de 'results'."""
    next_url = url
    while next_url:
        resp = session.get(next_url, timeout=60)
        if resp.status_code >= 400:
            raise RuntimeError(f"Falha HTTP {resp.status_code} ao acessar {next_url}: {resp.text[:500]}")
        data = resp.json()
        results = data.get("results", [])
        for item in results:
            yield item
        next_url = data.get("next")


def fetch_endpoint_records(name: str, url: str) -> List[dict]:
    """Baixa todos os registros de um endpoint SWAPI."""
    session = build_session_with_retry()
    items = list(fetch_paginated(session, url))
    print(f"[{name}] Registros baixados: {len(items)}")
    return items


# ========== NORMALIZAÇÃO DE COLUNAS ==========
def strip_accents(text: str) -> str:
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch)
    )


def sanitize_bq_field_name(name: str) -> str:
    """
    Normaliza nome de coluna para ser compatível com BigQuery:
    - remove acentos
    - to lower
    - troca espaços e separadores por underscore
    - remove caracteres inválidos
    - garante que não começa com dígito (prefixa com '_' se necessário)
    """
    if not isinstance(name, str):
        name = str(name)

    name = strip_accents(name)
    name = name.lower().strip()

    # substitui separadores por underscore
    name = re.sub(r"[^\w]+", "_", name, flags=re.UNICODE)  # qualquer não "word"

    # BigQuery: deve começar com letra ou underscore
    if re.match(r"^\d", name):
        name = "_" + name

    # remove underscores duplos/terminais
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")

    # fallback se esvaziou
    if not name:
        name = "_col"

    return name


def deduplicate_columns(cols: List[str]) -> List[str]:
    """Garante unicidade de nomes de colunas (col, col_1, col_2, ...)."""
    seen = {}
    result = []
    for c in cols:
        base = c
        idx = seen.get(base, 0)
        if idx == 0 and base not in result:
            result.append(base)
            seen[base] = 1
        else:

            while True:
                candidate = f"{base}_{idx}"
                idx += 1
                if candidate not in result:
                    result.append(candidate)
                    seen[base] = idx
                    break
    return result


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica json_normalize (prévio) -> sanitiza nomes -> deduplica."""
    new_cols = [sanitize_bq_field_name(c) for c in df.columns]
    new_cols = deduplicate_columns(new_cols)
    df = df.copy()
    df.columns = new_cols
    return df


# ========== CONVERSÃO DE TIPOS PARA STRING ==========
def to_json_string_if_complex(value):
    """Se for list/dict, converte para JSON string (preserva acentos)."""
    if isinstance(value, (list, dict)):
        try:
            return json.dumps(value, ensure_ascii=False)
        except Exception:
            return str(value)
    return value


def coerce_dataframe_to_strings(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converte todos os valores:
    - list/dict -> JSON string
    - outros tipos -> string (preservando NULL)
    """
    df = df.copy()
    for col in df.columns:
        # Converte listas/dicts para JSON string
        df[col] = df[col].map(to_json_string_if_complex)


        df[col] = df[col].astype("object")
        df[col] = df[col].where(df[col].isna(), df[col].map(lambda x: str(x)))
    return df


# ========== BIGQUERY ==========
def ensure_dataset(client: bigquery.Client, project_id: str, dataset_id: str):
    ds_ref = bigquery.DatasetReference(project_id, dataset_id)
    try:
        client.get_dataset(ds_ref)
    except NotFound:
        ds = bigquery.Dataset(ds_ref)
        ds.location = "us-east1"
        client.create_dataset(ds)
        print(f"Dataset criado: {project_id}.{dataset_id}")


def save_dataframe_to_bq(
    df: pd.DataFrame,
    client: bigquery.Client,
    project_id: str,
    dataset_id: str,
    table_name: str,
    enforce_all_string: bool = True,
    write_disposition: str = "WRITE_TRUNCATE",
):
    ensure_dataset(client, project_id, dataset_id)
    table_id = f"{project_id}.{dataset_id}.{table_name}"

    job_config = bigquery.LoadJobConfig(write_disposition=write_disposition)

    if enforce_all_string:

        schema = [bigquery.SchemaField(col, "STRING") for col in df.columns]
        job_config.schema = schema


        df = df.astype("object")


    load_job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    result = load_job.result()
    print(f"Gravou {result.output_rows} linhas em {table_id} (write_disposition={write_disposition}).")


# ========== PIPELINE ==========
def process_endpoint(name: str, url: str):

    records = fetch_endpoint_records(name, url)
    if not records:
        print(f"[{name}] Sem dados.")
        return


    df = pd.json_normalize(records, sep="__")

    # Normalizar nomes de colunas
    df = normalize_columns(df)


    df["dt_ingestao_utc"] = datetime.now(timezone.utc).isoformat()

    # Converter tudo para STRING (listas/dicts -> JSON string, nulos preservados)
    if ENFORCE_ALL_STRING:
        df = coerce_dataframe_to_strings(df)

    # 6) Salvar no BigQuery
    table_name = f"{TABLE_PREFIX}_{name}"
    save_dataframe_to_bq(
        df=df,
        client=bq_client,
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        table_name=table_name,
        enforce_all_string=ENFORCE_ALL_STRING,
        write_disposition=WRITE_DISPOSITION,
    )


def run():
    print(f"Início: {datetime.now().isoformat(timespec='seconds')}")
    for name, url in ENDPOINTS.items():
        try:
            print("=" * 90)
            print(f"Processando endpoint: {name} -> {url}")
            process_endpoint(name, url)
        except Exception as e:
            print(f"[{name}] ERRO: {e}")
    print(f"Fim: {datetime.now().isoformat(timespec='seconds')}")

run()

Início: 2026-03-02T20:21:02
Processando endpoint: films -> https://swapi.dev/api/films/
[films] Registros baixados: 6
Dataset criado: elet-dados-pessoas-dev.ds_ti_swapi_raw
Gravou 6 linhas em elet-dados-pessoas-dev.ds_ti_swapi_raw.tb_ti_swapi_raw_films (write_disposition=WRITE_TRUNCATE).
Processando endpoint: people -> https://swapi.dev/api/people/
[people] Registros baixados: 82
Gravou 82 linhas em elet-dados-pessoas-dev.ds_ti_swapi_raw.tb_ti_swapi_raw_people (write_disposition=WRITE_TRUNCATE).
Processando endpoint: planets -> https://swapi.dev/api/planets/
[planets] Registros baixados: 60
Gravou 60 linhas em elet-dados-pessoas-dev.ds_ti_swapi_raw.tb_ti_swapi_raw_planets (write_disposition=WRITE_TRUNCATE).
Processando endpoint: species -> https://swapi.dev/api/species/
[species] Registros baixados: 37
Gravou 37 linhas em elet-dados-pessoas-dev.ds_ti_swapi_raw.tb_ti_swapi_raw_species (write_disposition=WRITE_TRUNCATE).
Processando endpoint: starships -> https://swapi.dev/api/starships/

### Top 5 personagens que apareceu em mais filmes de Star Wars

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "elet-dados-pessoas-dev"
DATASET_ID = "ds_ti_swapi_raw"

bq = bigquery.Client(project=PROJECT_ID)

query = rf"""
WITH base_film AS (
  SELECT
    title,
    -- characters é STRING com array JSON; convertemos cada elem para STRING "pura"
    JSON_EXTRACT_SCALAR(elem, '$') AS character_url,
    url AS film_url
  FROM `{PROJECT_ID}.{DATASET_ID}.tb_ti_swapi_raw_films`
  CROSS JOIN UNNEST(JSON_EXTRACT_ARRAY(characters)) AS elem
),
exploded AS (
  SELECT
    CAST(REGEXP_EXTRACT(character_url, r'/(\d+)(?:/)?$') AS INT64) AS name_id,
    CAST(REGEXP_EXTRACT(film_url,      r'/(\d+)(?:/)?$') AS INT64) AS film_id,
    title
  FROM base_film
),
film AS (
  SELECT
    title,
    film_id,
    name_id
  FROM exploded
),
base_people AS (
  SELECT
    CAST(REGEXP_EXTRACT(url, r'/(\d+)(?:/)?$') AS INT64) AS name_id,
    name
  FROM `{PROJECT_ID}.{DATASET_ID}.tb_ti_swapi_raw_people`
),
base_counts AS (
  SELECT
    b.name AS personagem,
    COUNT(DISTINCT a.film_id) AS qtd_filmes
  FROM film AS a
  LEFT JOIN base_people AS b
    ON a.name_id = b.name_id
  GROUP BY personagem
)
SELECT
  personagem,
  qtd_filmes
FROM base_counts
QUALIFY ROW_NUMBER() OVER (ORDER BY qtd_filmes DESC) <= 5
"""


df_exploded = bq.query(query).result().to_dataframe()
df_exploded.head()

,personagem,qtd_filmes
0,C-3PO,6
1,R2-D2,6
2,Obi-Wan Kenobi,6
3,Yoda,5
4,Palpatine,5


In [ ]:
# dataframe: df_exploded
# uuid: ca6aa1a7-644e-438d-ffc4-d0939ce64878
# output_variable:
# config_str:

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='df_exploded', uuid='ca6aa1a7-644e-438d-ffc4-d0939ce64878')

<IPython.core.display.Javascript object>

 ### Planetas mais quentes do universo de Star Wars

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "elet-dados-pessoas-dev"
DATASET_ID = "ds_ti_swapi_raw"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""
  SELECT
    name
  FROM `{PROJECT_ID}.{DATASET_ID}.tb_ti_swapi_raw_planets`
  where climate = "hot"

"""


df_planets = bq.query(query).result().to_dataframe()
df_planets.head()

,name
0,Mustafar
1,Saleucami
2,Rodia


### Top 6 naves espaciais mais rápida do universo de Star Wars

In [26]:
from google.cloud import bigquery

PROJECT_ID = "elet-dados-pessoas-dev"
DATASET_ID = "ds_ti_swapi_raw"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""

WITH
 starships AS (

  SELECT
    name,
    cast(hyperdrive_rating as FLOAT64) as hyperdrive_rating,
    ROW_NUMBER() OVER (ORDER BY hyperdrive_rating ASC) as ranking

  FROM `{PROJECT_ID}.{DATASET_ID}.tb_ti_swapi_raw_starships`
  WHERE hyperdrive_rating <= '1.0'
 )

  SELECT
    name,
    hyperdrive_rating

  FROM starships
  WHERE ranking <=6

"""


df_starships = bq.query(query).result().to_dataframe()
df_starships.head(6)

,name,hyperdrive_rating
0,Millennium Falcon,0.5
1,Naboo star skiff,0.5
2,Republic Assault ship,0.6
3,J-type diplomatic barge,0.7
4,H-type Nubian yacht,0.9
5,X-wing,1.0


In [27]:
# dataframe: df_starships
# uuid: f2de4e12-18a0-4338-9d12-08cbe4860d21
# output_variable:
# config_str:

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='df_starships', uuid='f2de4e12-18a0-4338-9d12-08cbe4860d21')

<IPython.core.display.Javascript object>

###  Arma mais poderosa do universo de Star Wars:
##### Não tem um campo que diz exatamente o poder de fogo de cada nave ou arma em especifico, mas com base no custo de credito e na classe da nave cheguei a conclusão que a arma mais poderosa é a **Death Star**.

In [29]:
from google.cloud import bigquery

PROJECT_ID = "elet-dados-pessoas-dev"
DATASET_ID = "ds_ti_swapi_raw"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""

  SELECT
    name,

  CAST(case
      when cost_in_credits = 'n/a' or cost_in_credits = 'unknown'         then '0'
      else cost_in_credits                                                end
      as INT64) as cost_in_credits,

  starship_class

  FROM `{PROJECT_ID}.{DATASET_ID}.tb_ti_swapi_raw_starships`
  ORDER BY cost_in_credits desc

"""


df_starships_power = bq.query(query).result().to_dataframe()
df_starships_power.head()

,name,cost_in_credits,starship_class
0,Death Star,1000000000000,Deep Space Mobile Battlestation
1,Executor,1143350000,Star dreadnought
2,Star Destroyer,150000000,Star Destroyer
3,Trade Federation cruiser,125000000,capital ship
4,Calamari Cruiser,104000000,Star Cruiser
